# OpenAI Responses API

> Translate Responses API payloads, completed responses, and stream events to and from fastllm's shared message types

In [1]:
#| default_exp openai_responses

## Imports

In [2]:
#| export
import json
from collections import Counter
from fastcore.utils import *
from fastcore.meta import *
from fasttransport.errors import api_error_from_event

from fastllm.types import *
from aidialog.msg_parts import (Completion, Part, PartType, Msg, Text, Thinking, Refusal, ToolUse, ToolResult, tool_text, MdStr,
    InputImage, InputAudio, InputVideo, InputFile, mk_tool_res_msg, sys_text, part_txt, data_url)
from fastllm.streaming import *
from fastllm.streaming import mk_acollect_stream

In [3]:
#| hide
from fasttransport.core import AsyncHttpCli
from fastcore.test import *
from fastcore.funccall import get_schema
from cachy.core import enable_cachy, disable_cachy, doms
from aidialog.msg_parts import fmt2hist, hist2fmt
from fastllm.acomplete import acomplete, defaults


In [4]:
enable_cachy(doms=doms+sample_doms, hdrs=('content-type',))


In [5]:
#| hide
defaults.reasoning_effort = 'l'

The examples use `oai_call` to POST a request body to `/responses`. It calls `provider_req` through `AsyncHttpCli`, as `acomplete` does. An ordinary call returns the response object. With `stream=True`, it returns an event iterator.

In [6]:
oai_cli = AsyncHttpCli('https://api.openai.com/v1', base_headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
async def oai_call(**body): return await provider_req(oai_cli, '/responses', body, stream=body.get('stream', False))

## Normalize

The normalizers convert provider responses to `ToolUse`, `Usage`, and message parts. `mk_completion` combines those results into a `Completion`.

### Tool Calls

`norm_tool_call` distinguishes client function calls from tools the server ran. Function arguments arrive as JSON text and become a dict. Hosted calls use `server=True`. Their arguments come from `arguments` or `action`, and `output` supplies result text when present. An `md` flag preserves a Markdown result as `MdStr`.

In [7]:
def lite_mk_func(f):
    if isinstance(f, dict): return f
    return {'type':'function', 'function':get_schema(f, pname='parameters')}

In [8]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [9]:
sch = lite_mk_func(simple_add)
sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

In [10]:
#| export
def norm_tool_call(item):
    typ = item.get("type", "")
    if typ == "function_call":
        raw = {k:v for k,v in item.items() if k not in ("call_id","name","arguments")}
        return ToolUse(id=item.get("call_id") or item["id"], name=item["name"], arguments=json.loads(item["arguments"]), raw=raw)
    if typ.endswith("_call"):  # a call the server ran: OpenAI's hosted items carry an `action`, a Responses server's `server_tool_call` its `arguments` and `output` (`md` when that output is markdown)
        raw = {k:v for k,v in item.items() if k not in ("id","name","arguments","output","action","md")}
        args = json.loads(item["arguments"]) if "arguments" in item else item.get("action", {})
        out = item.get("output")
        return ToolUse(id=item.get("id",""), name=item.get("name") or typ.removesuffix("_call"), arguments=args, server=True, text=MdStr(out) if item.get("md") else out, raw=raw)

def norm_tool_calls(resp):
    "Extract Responses API tool call items as normalized tool calls."
    out = []
    for item in resp.get('output', []):
        if tc := norm_tool_call(item): out.append(tc)
    return out

User defined:

In [11]:
resp = await oai_call(model=astra, input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.",
    tools=[{"type": "function", **sch['function']}], parallel_tool_calls=True)
items = [o for o in obj2dict(resp)['output'] if o['type']=='function_call']
calls = norm_tool_calls(resp)
test_eq(len(calls), 2)
test_eq(L(calls).attrgot('id'), [o['call_id'] for o in items])
calls

[ToolUse(raw={'id': 'fc_0a5082d16bcd5292006abaf994ad9487d095ece2e656b70a3f', 'type': 'function_call', 'status': 'completed'}, cache_control=None, id='call_actlNu1ETWxjH9t65ZVPF0gs', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, text=None),
 ToolUse(raw={'id': 'fc_0a5082d16bcd5292006abaf994ada487d0a9e63fb1aab3432a', 'type': 'function_call', 'status': 'completed'}, cache_control=None, id='call_fTqujctRVTVN4nqdcFIYMZed', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, text=None)]

Web search:

In [12]:
resp = await oai_call(model=luna, input="What is the weather in San Francisco today?", tools=[{"type": "web_search_preview"}])
norm_tool_calls(resp)

[ToolUse(raw={'type': 'web_search_call', 'status': 'completed'}, cache_control=None, id='ws_0314c2448fcfdb5f006abb081547b487d0ba9c59633fb2dae6', name='web_search', arguments={'type': 'search', 'queries': ['weather: San Francisco, CA'], 'query': 'weather: San Francisco, CA'}, server=True, text=None)]

File vector search:

In [13]:
norm_tool_calls({"output": [dict(type="file_search_call", id="fs_abc123", status="completed", queries=["quarterly revenue"],
    results=[{"file_id": "file-xyz", "text": "Revenue was $1M"}])]})

[ToolUse(raw={'type': 'file_search_call', 'status': 'completed', 'queries': ['quarterly revenue'], 'results': [{'file_id': 'file-xyz', 'text': 'Revenue was $1M'}]}, cache_control=None, id='fs_abc123', name='file_search', arguments={}, server=True, text=None)]

MCP:

In [14]:
norm_tool_calls({"output": [dict(type="function_call", call_id="call_mcp1", id="fc_mcp1", name="mcp__weather__get_forecast",
    arguments='{"city": "Istanbul"}', status="completed")]})

[ToolUse(raw={'type': 'function_call', 'id': 'fc_mcp1', 'status': 'completed'}, cache_control=None, id='call_mcp1', name='mcp__weather__get_forecast', arguments={'city': 'Istanbul'}, server=False, text=None)]

### Usage

`norm_usage` accepts both Responses token names and the corresponding Chat names. It records input, output, total, cached, and reasoning tokens. Hosted calls also contribute counts in `raw['server_tool_use']`. Web searches contribute to `search_queries`. A response without usage returns `None`.

In [15]:
#| export
def norm_usage(resp):
    "Normalize OpenAI usage shape(s)."
    if not (usg:=resp.get("usage")): return None
    pt = int(usg.get("prompt_tokens", usg.get("input_tokens", 0)) or 0)
    ct = int(usg.get("completion_tokens", usg.get("output_tokens", 0)) or 0)
    tt = int(usg.get("total_tokens", pt + ct) or (pt + ct))
    pd = usg.get("prompt_tokens_details") or usg.get("input_tokens_details") or {}
    cd = usg.get("completion_tokens_details") or usg.get("output_tokens_details") or {}
    cached = int(pd.get("cached_tokens", 0) or 0)
    reasoning = int(cd.get("reasoning_tokens", 0) or 0)
    server_tool_use = dict(Counter(o['type'] for o in resp.get('output', []) if o.get('type') != 'function_call' and o.get('type', '').endswith('_call')))
    if server_tool_use: usg['server_tool_use'] = server_tool_use
    return Usage(prompt_tokens=pt, completion_tokens=ct, total_tokens=tt,
        cached_tokens=cached, reasoning_tokens=reasoning, search_queries=server_tool_use.get('web_search_call', 0), raw=usg)

Text response:

In [16]:
resp = await oai_call(model=luna, input=[{"role": "user", "content": "Say hi"}])
norm_usage(resp)

Usage(prompt_tokens=8, completion_tokens=8, total_tokens=16, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 8, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 8, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 16})

Tool call response:

In [17]:
resp = await oai_call(model=luna, input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.", tools=[{"type": "function", **sch['function']}])
norm_usage(resp)

Usage(prompt_tokens=65, completion_tokens=56, total_tokens=121, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 65, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 56, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 121})

Server tools:

In [18]:
resp = await oai_call(model=luna, input="What is the weather in San Francisco today?", tools=[{"type": "web_search_preview"}])
norm_usage(resp)

Usage(prompt_tokens=4710, completion_tokens=91, total_tokens=4801, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=30, search_queries=1, raw={'input_tokens': 4710, 'input_tokens_details': {'cache_write_tokens': 4383, 'cached_tokens': 0}, 'output_tokens': 91, 'output_tokens_details': {'reasoning_tokens': 30}, 'total_tokens': 4801, 'server_tool_use': {'web_search_call': 1}})

### Finish Reason

`completed` becomes `stop` and `failed` becomes `content_filter`. An `incomplete` response says why in `incomplete_details.reason`: `max_output_tokens` becomes `length` and `content_filter` stays `content_filter`, so a filtered transcription is not mistaken for a short budget. A successful response with a client tool call becomes `tool_calls`. Server tools don't require a client response.

In [19]:
#| export
def norm_finish(resp, tcs=None):
    "Canonicalize finish_reason to OpenAI Chat values: stop, tool_calls, length, content_filter."
    reason = resp.get("status")
    if reason == 'incomplete': reason = nested_idx(resp, 'incomplete_details', 'reason') or reason
    mp = dict(completed=FinishReason.stop, incomplete=FinishReason.length, max_output_tokens=FinishReason.length,
              content_filter=FinishReason.content_filter, failed=FinishReason.content_filter)
    r =  mp.get(reason, reason)
    return FinishReason.tool_calls if r==FinishReason.stop and any(~L(tcs).attrgot('server')) else r 

In [20]:
test_eq(norm_finish(dict(status='completed'), ToolUse('123', 'f')), 'tool_calls')
test_eq(norm_finish(dict(status='completed'), ToolUse('123', 'web_search', server=True)), 'stop')

### Non-stream Completion

`norm_parts` reads each item in `output`. Message items contain `output_text` or `refusal` blocks. These supply the `text` and `refusal` fields respectively. Reasoning items contain a `summary` array whose entries supply `text`.

This implementation indexes those required fields directly. Missing fields raise `KeyError`; there is no empty-string fallback here. It ignores other message-content types. It reads every reasoning summary entry without checking a `summary_text` discriminator.


The hosted-search examples report a `web_search_call` and put source citations on message text. `search_hits` collects those citations by URL. It fills empty server-tool results with `title: url` lines wrapped in `MdStr`. Existing result text stays unchanged. This provides readable results for display and stored replies.

In [21]:
#| export
def search_hits(parts):
    "Fill empty server-tool results with the message's cited sources as `title: url` lines"
    hits = {c['url']: c.get('title', '') for p in parts if isinstance(p, Text) for c in p.citations or [] if c.get('url')}
    for p in parts:
        if isinstance(p, ToolUse) and p.server and not p.text: p.text = MdStr('\n'.join(f"{t}: {u}" for u,t in hits.items()))
    return parts

In [22]:
#| export
def norm_parts(resp):
    parts = []
    for item in resp["output"]:
        if (typ:=item["type"]) == "message":
            for c in item["content"]:
                if (ctyp := c["type"]) == "output_text": parts.append(Text(c['text'], citations=list(c.get('annotations') or []), raw=c))
                elif ctyp == "refusal": parts.append(Refusal(c['refusal'], raw=c))
        elif typ == "reasoning":
            for s in item["summary"]: parts.append(Thinking(s['text'], raw=item))
        elif typ == "function_call" or typ.endswith("_call"): parts.append(norm_tool_call(item))
    return search_hits(parts)

`raw_msg` retains the response's original `output` items in `Msg.raw`. Replaying those items preserves reasoning and hosted-tool data without `previous_response_id`. Keeping them intact matters when [manually continuing a reasoning conversation](https://developers.openai.com/api/docs/guides/reasoning#keeping-reasoning-items-in-context).

`denorm_assistant` returns the retained list rather than reconstructing its items.

In [23]:
#| export
def raw_msg(resp):
    "Return the response output list for assistant replay, without copying"
    return resp['output']

Register the provider's normalizers with `api_registry`. `mk_completion` looks them up to build the shared `Completion` object. `raw_msg` supplies the assistant message's replay data.

In [24]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage, raw_msg=raw_msg)
api_registry.register('openai', **norm_funcs)

`Completion.response_id` supports complete responses and collected streams. It first checks `raw['response_id']`. For the `openai` API, it then checks `raw['id']` and the response IDs in stored stream events.

For `vendor_name='codex'`, it returns `None`. That transport doesn't support `previous_response_id`; callers must replay history.

In [25]:
#| export
@patch(as_prop=True)
def response_id(self:Completion):
    "Provider continuation identifier."
    if self.vendor_name == 'codex': return
    if rid := self.raw.get('response_id'): return rid
    if self.api_name != 'openai': return
    if rid := self.raw.get('id'): return rid
    for d in reversed(self.raw.get('deltas', [])):
        if rid := nested_idx(d.raw, 'response', 'id'): return rid

In [26]:
live_response = Completion('', Msg('assistant', []), raw=dict(response_id='resp_live'))
test_eq(live_response.response_id, 'resp_live')
codex_response = Completion('', Msg('assistant', []), api_name='openai', vendor_name='codex', raw=dict(id='resp_codex'))
test_eq(codex_response.response_id, None)
dict(live=live_response.response_id, codex=codex_response.response_id)

{'live': 'resp_live', 'codex': None}

#### Text

In [27]:
mn,api_name,vnd_nm = luna, 'openai', 'openai'
resp = await oai_call(model=mn, input="Say hi")
c = mk_completion(resp, mn, api_name, vnd_nm)
c.message.content, c.finish_reason

([Text(raw={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'Hi!'}, cache_control=None, text='Hi!', citations=[])],
 <finish_reason.stop: 'stop'>)

#### Tool call

In [28]:
resp = await oai_call(model=mn, input="What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.", tools=[{"type": "function", **sch['function']}])
c = mk_completion(resp, mn, api_name, vnd_nm)
test_eq(c.finish_reason, 'tool_calls')
test_eq(len(c.message.content), 2)
c.tool_calls, c.message.content

([ToolUse(raw={'id': 'fc_03e4debb7e4b5259006abb081926bc87d0baea1aae0705e30c', 'type': 'function_call', 'status': 'completed'}, cache_control=None, id='call_uHmSAUWQXcthEvktHHcYr6Wa', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, text=None),
  ToolUse(raw={'id': 'fc_03e4debb7e4b5259006abb081926d487d0bd9c31f5aa11a1e7', 'type': 'function_call', 'status': 'completed'}, cache_control=None, id='call_pU55HyIoSdN7ViiKVFZRh9cn', name='simple_add', arguments={'a': 10, 'b': 20}, server=False, text=None)],
 [ToolUse(raw={'id': 'fc_03e4debb7e4b5259006abb081926bc87d0baea1aae0705e30c', 'type': 'function_call', 'status': 'completed'}, cache_control=None, id='call_uHmSAUWQXcthEvktHHcYr6Wa', name='simple_add', arguments={'a': 3, 'b': 5}, server=False, text=None),
  ToolUse(raw={'id': 'fc_03e4debb7e4b5259006abb081926d487d0bd9c31f5aa11a1e7', 'type': 'function_call', 'status': 'completed'}, cache_control=None, id='call_pU55HyIoSdN7ViiKVFZRh9cn', name='simple_add', arguments={'a': 10, 'b': 20

This search example checks the server-tool flag, finish reason, query count, and citation-derived result text.


In [29]:
resp = await oai_call(model=mn, input="What is the latest stable Python release? Cite your sources.", tools=[{"type": "web_search_preview"}])
c = mk_completion(resp,mn,api_name,vnd_nm)
c.tool_calls, next(p.text for p in c.message.content if p.type == 'text')[:100]

([ToolUse(raw={'type': 'web_search_call', 'status': 'completed'}, cache_control=None, id='ws_0a24e3a21d164a5f006abb08d8447c87d08369ac7ed4578245', name='web_search', arguments={'type': 'search', 'queries': ['site:python.org/downloads/ latest stable Python release', 'site:python.org/downloads/release Python 3 latest release'], 'query': 'site:python.org/downloads/ latest stable Python release'}, server=True, text='Download Python | Python.org: https://www.python.org/downloads/'),
  ToolUse(raw={'type': 'web_search_call', 'status': 'completed'}, cache_control=None, id='ws_0a24e3a21d164a5f006abb08dae5a887d08be004b0e6d2b5f9', name='web_search', arguments={'type': 'open_page', 'url': 'https://www.python.org/downloads/release/python-3147/'}, server=True, text='Download Python | Python.org: https://www.python.org/downloads/')],
 'As of **September 29, 2026**, the latest stable Python release is **Python 3.14.7**, released on **A')

In [30]:
test_eq(c.tool_calls[0].server, True)
test_eq(c.finish_reason, 'stop')
test_eq(c.usage.search_queries, len([o for o in resp['output'] if o['type']=='web_search_call']))
assert ': http' in c.tool_calls[0].text, c.tool_calls[0].text
c.tool_calls[0].text

'Download Python | Python.org: https://www.python.org/downloads/'

The refusal example uses a fixture because a live refusal is hard to trigger reliably.

In [31]:
c = mk_completion({"model": luna, "status": "completed", "output": [dict(type="message", role="assistant", content=[{"type": "refusal", "refusal": "I can't help with that request."}])], "usage": dict(input_tokens=10, output_tokens=5, total_tokens=15)}, mn, api_name, vnd_nm)
c.message.content

[Refusal(raw={'type': 'refusal', 'refusal': "I can't help with that request."}, cache_control=None, text="I can't help with that request.")]

### Streaming

`norm_sse_event` converts provider events to `Delta` objects. The shared streaming module uses `PartAccum` to collect them. It yields text, thinking, and citation updates before the final `Completion`.

In [32]:
print_stream = PrintStream()

Responses sends complete tool-call arguments in `response.output_item.done`. This converter reads the completed call instead of collecting argument fragments.

For comparison, a provider that sends argument fragments uses `arguments={'_delta': ...}` in its `ToolUse` deltas:

```py
ToolUse(id=tc.get("id",""), name=fn.get("name",""), arguments={'_delta': fn.get("arguments")}, raw=raw)
```

Text, reasoning text, reasoning summaries, and annotations have separate event branches below. Completion events supply finish reason and usage. An `error` event raises the transport's API error.

In [33]:
#| export
def norm_sse_event(ev, **kwargs):
    "Normalize OpenAI Responses API stream event into Delta."
    ev = obj2dict(ev)
    typ = ev.get("type")
    if typ == "response.output_text.delta":            return Delta(text=ev.get("delta"), raw=ev, **kwargs)
    if typ == "response.reasoning_text.delta":         return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.reasoning_summary_text.delta": return Delta(thinking=ev.get("delta",""), raw=ev, **kwargs)
    if typ == "response.output_text.annotation.added": return Delta(citations=[ev.get("annotation")], raw=ev, **kwargs)
    if typ == "response.output_item.done":
        item = ev.get("item", {})
        itype = item.get("type", "")
        if itype == "function_call" or itype.endswith("_call"): return Delta(tool_calls=[norm_tool_call(item)], raw=ev, **kwargs)
        return Delta(raw=ev, **kwargs)
    if typ in ("response.completed", "response.incomplete"):
        resp = ev.get("response", {})
        return Delta(finish_reason=norm_finish(resp), usage=norm_usage(resp), raw=ev, **kwargs)
    if typ == "error": raise api_error_from_event(ev)

The collector asks `delta_index_fn` which accumulated part to update. The function returns that part's index and the index to retain for the next event.

Here `d` is a `Delta`. Responses supplies its `output_index` in `d.raw`. This implementation uses that index directly and doesn't need `typ`, `last_typ`, or `last_idx`.

In [34]:
#| export
def delta_index_fn(d, typ, last_typ, last_idx):
    'Returns accumulation index for current delta and updated last idx'
    idx = nested_idx(d, 'raw', 'output_index')
    return idx, idx

`collate_raw` collects the items from `response.output_item.done` events in arrival order. It doesn't use the terminal event's output list. The Codex backend can leave that list empty.

In [35]:
#| export
def collate_raw(deltas):
    "The output items rebuilt from a stream's events"
    return [d.raw['item'] for d in deltas if d.raw.get('type') == 'response.output_item.done']

`acollect_stream` combines event normalization with the shared collector. Before yielding the final `Completion`, it fills empty server-tool results from citations.

In [36]:
acollect_stream = partial(mk_acollect_stream, index_fn=delta_index_fn, api_name='openai')

In [37]:
#| export
@delegates(mk_acollect_stream, but=['index_fn', 'api_name'])
async def acollect_stream(resp, **kwargs):
    res = mk_acollect_stream(norm_and_yield(resp, norm_sse_event), index_fn=delta_index_fn, api_name='openai', **kwargs)
    async for o in res:
        if isinstance(o, Completion): search_hits(o.message.content)
        yield o

In [38]:
norm_funcs = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    acollect_stream=acollect_stream, raw_msg=raw_msg, collate_raw=collate_raw)
api_registry.register('openai', **norm_funcs)

In [39]:
api_registry['openai']

namespace(finalize_usage=<function fastcore.imports.noop(x=None, *args, **kwargs)>,
          raw_msg=<function __main__.raw_msg(resp)>,
          collate_raw=<function __main__.collate_raw(deltas)>,
          norm_tool_calls=<function __main__.norm_tool_calls(resp)>,
          norm_parts=<function __main__.norm_parts(resp)>,
          norm_finish=<function __main__.norm_finish(resp, tcs=None)>,
          norm_usage=<function __main__.norm_usage(resp)>,
          acollect_stream=<function __main__.acollect_stream(resp, *, model=None, vendor_name=None, stop_callables=None)>)

Let's compare streamed and non-streamed results for text, thinking, tools, and web search.

#### Text

In [40]:
mn,inp = luna,'Hi!'
resp = await oai_call(model=mn,input=inp)
comp = mk_completion(resp, mn, api_name, vnd_nm)
comp

Hi! How can I help?

::: details

- model: `gpt-6-luna`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=8, completion_tokens=26, total_tokens=34, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=13, search_queries=0, raw={'input_tokens': 8, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 26, 'output_tokens_details': {'reasoning_tokens': 13}, 'total_tokens': 34})`

:::

In [41]:
resp = await oai_call(model=mn, input=inp, stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

Hi! How can I help?

In [42]:
resp = await oai_call(model=mn, input=inp, stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

Hi! How can I help?

In [43]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)
comp.usage, o.usage

(Usage(prompt_tokens=8, completion_tokens=26, total_tokens=34, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=13, search_queries=0, raw={'input_tokens': 8, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 26, 'output_tokens_details': {'reasoning_tokens': 13}, 'total_tokens': 34}),
 Usage(prompt_tokens=8, completion_tokens=11, total_tokens=19, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 8, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 11, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 19}))

#### Thinking

In [44]:
mn,inp,eff = luna,'What is 31231231 * 12312?',{"effort": "low", "summary": "auto"}
resp = await oai_call(model=mn,input=inp,reasoning=eff)
comp = mk_completion(resp, mn, api_name, vnd_nm)

In [45]:
comp


::: details
## Thinking

**Calculating sums and totals**

I need to calculate something: it begins with multiplying 31,231,231 by 12,312. Then, I calculate 12000 plus 312, which gives me 12,312. From there, I find the total as 374,774,772,000, and then add 312 multiplied out. The overall sum amounts to 9,744,144,072, leading me to a final total of 384,518,916,072. I’ll need to double-check that to ensure accuracy!

:::

31231231 × 12312 = **384518916072**

::: details

- model: `gpt-6-luna`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=17, completion_tokens=100, total_tokens=117, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=80, search_queries=0, raw={'input_tokens': 17, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 100, 'output_tokens_details': {'reasoning_tokens': 80}, 'total_tokens': 117})`

:::

In [46]:
resp = await oai_call(model=mn,input=inp,reasoning=eff,stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠31231231 × 12312 = **384,518,916,072**.

In [47]:
test_eq('thinking' in L(comp.message.content).attrgot('type'), True)
test_eq('thinking' in L(o.message.content).attrgot('type'), True)

In [48]:
test_eq(comp.finish_reason, o.finish_reason)
test_eq(comp.tool_calls, o.tool_calls)

#### Tool call

In [49]:
mn,inp = luna,'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'
tools=[{"type": "function", **sch['function']}]
resp = await oai_call(model=mn,input=inp,tools=tools)
comp = mk_completion(resp, mn, api_name, vnd_nm)

In [50]:
comp



🔧 simple_add({'a': 3, 'b': 5})


🔧 simple_add({'a': 10, 'b': 20})


::: details

- model: `gpt-6-luna`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=65, completion_tokens=56, total_tokens=121, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 65, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 56, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 121})`

:::

In [51]:
resp = await oai_call(model=mn, input=inp, tools=tools, stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)


- ⏳ `simple_add(a="3", b="5")` ⏳

- ⏳ `simple_add(a="10", b="20")` ⏳


In [52]:
def test_tcs(tcs1,tcs2):
    for tc1,tc2 in zip(tcs1,tcs2): 
        test_eq(tc1.name,tc2.name)
        test_eq(tc1.arguments,tc2.arguments)
        test_eq(tc1.server,tc2.server)

In [53]:
test_tcs(comp.tool_calls,o.tool_calls)

In [54]:
test_eq(len(comp.message.content), len(o.message.content))
test_eq(set(comp.message.content[0].raw.keys()), set(o.message.content[0].raw.keys()))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=65, completion_tokens=56, total_tokens=121, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 65, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 56, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 121}),
 Usage(prompt_tokens=65, completion_tokens=56, total_tokens=121, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 65, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 56, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 121}))

#### Web search

In [55]:
mn,inp,tools = luna,'What is the latest stable Python release? Cite your sources.',[{"type": "web_search_preview"}]
resp = await oai_call(model=mn,input=inp,tools=tools)
comp = mk_completion(resp, mn, api_name, vnd_nm)

In [56]:
comp



🔧 web_search({'type': 'search', 'queries': ['site:python.org/downloads/ latest stable Python release', 'site:python.org/downloads/release Python 3 latest release'], 'query': 'site:python.org/downloads/ latest stable Python release'})


🔧 web_search({'type': 'open_page', 'url': 'https://www.python.org/downloads/release/python-3147/'})
As of **September 29, 2026**, the latest stable Python release is **Python 3.14.7**, released on **August 5, 2026**. Python.org lists it as the newest release; Python 3.15 is still a pre-release. ([python.org](https://www.python.org/downloads/))

::: details

- model: `gpt-6-luna`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=17234, completion_tokens=222, total_tokens=17456, cached_tokens=4370, cache_creation_tokens=0, reasoning_tokens=139, search_queries=2, raw={'input_tokens': 17234, 'input_tokens_details': {'cache_write_tokens': 16, 'cached_tokens': 4370}, 'output_tokens': 222, 'output_tokens_details': {'reasoning_tokens': 139}, 'total_tokens': 17456, 'server_tool_use': {'web_search_call': 2}})`

:::

Same with streaming:

In [57]:
resp = await oai_call(model=mn, input=inp, tools=tools, stream=True)
async for o in acollect_stream(resp, vendor_name=vnd_nm): print_stream(o)

The latest stable Python release is **Python 3.14.7**. Python.org lists it as the latest Python 3 release and identifies it as a stable release. ([python.org](https://www.python.org/getit/source/?utm_source=openai))

In [58]:
cites = o.message.content[-1].citations
cites

[{'type': 'url_citation',
  'end_index': 215,
  'start_index': 145,
  'title': 'Python Source Releases | Python.org',
  'url': 'https://www.python.org/getit/source/?utm_source=openai'}]

In [59]:
o.message.content[-1].text[cites[0]['start_index']:]

'([python.org](https://www.python.org/getit/source/?utm_source=openai))'

In [60]:
test_eq(set(L(comp.tool_calls).attrgot('name')), set(L(o.tool_calls).attrgot('name')))
test_eq(comp.finish_reason, o.finish_reason)
comp.usage, o.usage

(Usage(prompt_tokens=17234, completion_tokens=222, total_tokens=17456, cached_tokens=4370, cache_creation_tokens=0, reasoning_tokens=139, search_queries=2, raw={'input_tokens': 17234, 'input_tokens_details': {'cache_write_tokens': 16, 'cached_tokens': 4370}, 'output_tokens': 222, 'output_tokens_details': {'reasoning_tokens': 139}, 'total_tokens': 17456, 'server_tool_use': {'web_search_call': 2}}),
 Usage(prompt_tokens=8632, completion_tokens=104, total_tokens=8736, cached_tokens=4386, cache_creation_tokens=0, reasoning_tokens=44, search_queries=1, raw={'input_tokens': 8632, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 4386}, 'output_tokens': 104, 'output_tokens_details': {'reasoning_tokens': 44}, 'total_tokens': 8736, 'server_tool_use': {'web_search_call': 1}}))

## Denormalize

The request converters turn shared messages and options into Responses API fields.

### Msgs

In [61]:
def mk_user_msg(txt): return Msg(role='user', content=[Text(txt)])

User messages become input items. Assistant messages replay their retained output items when available. Tool messages supply function-call outputs.

In [62]:
mn,inp = luna,'What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'
tools=[{"type": "function", **sch['function']}]
resp = await oai_call(model=mn, input=inp, tools=tools, stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass

In [63]:
comp



🔧 simple_add({'a': 3, 'b': 5})


🔧 simple_add({'a': 10, 'b': 20})


::: details

- model: `None`
- finish_reason: `tool_calls`
- usage: `Usage(prompt_tokens=64, completion_tokens=56, total_tokens=120, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 64, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 56, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 120})`

:::

In [64]:
trmsg = mk_tool_res_msg(comp.tool_calls, ('8', '30'))
trmsg

**Msg**

- role: `tool`

<contents>

**ToolResult** (`tool_result`)

8

::: details

- raw: `None`
- id: `call_AoBcYD6mLzVQckKfPb2iEM2n`
- name: `simple_add`
- arguments: `{'a': 3, 'b': 5}`
- server: `False`

:::

**ToolResult** (`tool_result`)

30

::: details

- raw: `None`
- id: `call_uLmb7KJ6wWrwJpT72DVPpxfA`
- name: `simple_add`
- arguments: `{'a': 10, 'b': 20}`
- server: `False`

:::

</contents>

In [65]:
#| export
def _sanid(id_str): return id_str[:64] # codex max 64 char limit

In [66]:
#| export
def denorm_tool_use(p:ToolUse):
    "Convert canonical tool_use Part back to OpenAI Responses function_call item."
    return dict(type='function_call', call_id=_sanid(p.id), name=p.name, arguments=json.dumps(p.arguments))

In [67]:
L(comp.message.content).map(denorm_tool_use)

[{'type': 'function_call', 'call_id': 'call_AoBcYD6mLzVQckKfPb2iEM2n', 'name': 'simple_add', 'arguments': '{"a": 3, "b": 5}'}, {'type': 'function_call', 'call_id': 'call_uLmb7KJ6wWrwJpT72DVPpxfA', 'name': 'simple_add', 'arguments': '{"a": 10, "b": 20}'}]

This first tool-result converter handles text. The exported version below also handles media.

In [68]:
def denorm_tool_result(p:ToolResult):
    "Convert canonical tool result back to OpenAI Responses function_call_output item."
    return dict(type='function_call_output', call_id=_sanid(p.id), output=tool_text(p.text))

In [69]:
#| export
def denorm_tool(m:Msg): return [denorm_tool_result(p) for p in m.content if isinstance(p, ToolResult)]

In [70]:
denorm_tool(trmsg)

[{'type': 'function_call_output',
  'call_id': 'call_AoBcYD6mLzVQckKfPb2iEM2n',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'call_uLmb7KJ6wWrwJpT72DVPpxfA',
  'output': '30'}]

This first user-message converter handles text. The exported version below adds media inputs.

In [71]:
def denorm_user(m:Msg):
    "Convert canonical user Msg to OpenAI Responses input message."
    parts = [{"type": "input_text", "text": p.text or ""} for p in m.content if isinstance(p, Text)]
    return dict(type="message", role="user", content=parts)

In [72]:
umsg = mk_user_msg(inp)
denorm_user(umsg)

{'type': 'message',
 'role': 'user',
 'content': [{'type': 'input_text',
   'text': 'What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'}]}

In [73]:
#| export
def denorm_assistant(m:Msg):
    "The output items as the API sent them, else the message's text and tool calls as items"
    if m.raw: return m.raw
    items = []
    for p in m.content:
        if isinstance(p, ToolUse): items.append(denorm_tool_use(p))
        elif isinstance(p, Text): items.append(dict(type="message", role="assistant", content=[dict(type="output_text", text=p.text or '')]))
    return items

In [74]:
denorm_assistant(comp.message)

[{'id': 'fc_05baf9759cda9345006abb0975c3e887d08ac24a2465a2c2ed',
  'type': 'function_call',
  'status': 'completed',
  'arguments': '{"a":3,"b":5}',
  'call_id': 'call_AoBcYD6mLzVQckKfPb2iEM2n',
  'name': 'simple_add'},
 {'id': 'fc_05baf9759cda9345006abb0975c3fc87d0b628f83ada181e14',
  'type': 'function_call',
  'status': 'completed',
  'arguments': '{"a":10,"b":20}',
  'call_id': 'call_uLmb7KJ6wWrwJpT72DVPpxfA',
  'name': 'simple_add'}]

In [75]:
#| export
def denorm_msgs(msgs:list[Msg]):
    res = []
    for m in msgs:
        if m.role == 'user':        res.append(denorm_user(m))
        elif m.role == 'assistant': res.extend(denorm_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_tool(m))
    return res

In [76]:
inputs = denorm_msgs([umsg, comp.message, trmsg])
test_eq(inputs[1:1+len(comp.message.raw)], comp.message.raw)
inputs

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20. Use the simple_add tool in parallel'}]},
 {'id': 'fc_05baf9759cda9345006abb0975c3e887d08ac24a2465a2c2ed',
  'type': 'function_call',
  'status': 'completed',
  'arguments': '{"a":3,"b":5}',
  'call_id': 'call_AoBcYD6mLzVQckKfPb2iEM2n',
  'name': 'simple_add'},
 {'id': 'fc_05baf9759cda9345006abb0975c3fc87d0b628f83ada181e14',
  'type': 'function_call',
  'status': 'completed',
  'arguments': '{"a":10,"b":20}',
  'call_id': 'call_uLmb7KJ6wWrwJpT72DVPpxfA',
  'name': 'simple_add'},
 {'type': 'function_call_output',
  'call_id': 'call_AoBcYD6mLzVQckKfPb2iEM2n',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'call_uLmb7KJ6wWrwJpT72DVPpxfA',
  'output': '30'}]

Rendering this turn with `hist2fmt` and reading it with `fmt2hist` discards `raw`. `denorm_assistant` then reconstructs plain `function_call` items. Their results become `function_call_output` items. The example submits that reconstructed history for another turn.

In [77]:
stored = fmt2hist(hist2fmt([comp.message, trmsg]))
stored_inputs = denorm_msgs([umsg, *stored, mk_user_msg('Now add 1 to each. Use simple_add.')])
test_eq([i['type'] for i in stored_inputs if i['type'].startswith('function_call')], ['function_call', 'function_call', 'function_call_output', 'function_call_output'])
resp = await oai_call(model=mn, input=stored_inputs, tools=tools)
[tc.arguments for tc in mk_completion(resp, mn, api_name, vnd_nm).tool_calls]


[{'a': 8, 'b': 1}, {'a': 30, 'b': 1}]

In [78]:
resp = await oai_call(model=mn, input=inputs, tools=tools, stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

3 + 5 = 8  
10 + 20 = 30

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=142, completion_tokens=19, total_tokens=161, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 142, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 19, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 161})`

:::

### Tool Schemas

In [79]:
#| export
def denorm_tool_schs(tools):
    "Convert canonical tools to OpenAI Responses format."
    out = []
    for t in tools:
        if isinstance(t, dict) and t.get('type')=='function' and 'name' in t and 'parameters' in t: out.append(t); continue
        fn = fn_schema(t)
        if fn is None: out.append(t); continue
        name, desc, params = fn
        out.append(dict(type='function', name=name, description=desc, parameters=params))
    return out

In [80]:
test_eq(denorm_tool_schs([sch]), [dict(type='function', name='simple_add', description='add numbers', parameters=sch['function']['parameters'])])
test_eq(denorm_tool_schs([{"type": "web_search_preview"}]), [{"type": "web_search_preview"}])
flat = dict(type='function', name='py', description='run code', parameters={'type':'object'}, strict=True)
test_eq(denorm_tool_schs([flat]), [flat])

### Tool Choice

In [81]:
#| export
def denorm_tool_choice(v):
    "Map canonical tool_choice to OpenAI Responses format."
    _tc_modes = {'auto', 'required', 'any', 'force', 'none', 'off', 'disabled'}
    if v is None: return None
    if v in _tc_modes: return v if v in ('auto','none','required') else dict(auto='auto', any='required', force='required', off='none', disabled='none')[v]
    return {'type': 'function', 'name': v}

Modes

In [82]:
test_eq(denorm_tool_choice('auto'), 'auto')
test_eq(denorm_tool_choice('required'), 'required')

Tool name

In [83]:
test_eq(denorm_tool_choice('simple_add'), {'type': 'function', 'name': 'simple_add'})

None

In [84]:
test_eq(denorm_tool_choice(None), None)

### Reasoning Effort

`denorm_reasoning` wraps a string in `{'effort': ...}`. An integer becomes `{'level': ...}` for a Responses server that resolves levels per model. `None` and other values pass through. This helper has no special `disable` option.

Anthropic uses a separate adaptive-thinking converter. Its model compatibility rules don't apply to this Responses helper.

In [85]:
#| export
def denorm_reasoning(v):
    "Map canonical reasoning_effort to OpenAI Responses reasoning param."
    if v is None: return None
    if isinstance(v, int): return {'level': v}  # a 0-3 level, for a Responses server that resolves it per model
    return {'effort': v} if isinstance(v, str) else v

In [86]:
test_eq(denorm_reasoning('low'), {'effort': 'low'})
test_eq(denorm_reasoning(None), None)

### Web Search Options

In [87]:
#| export
def denorm_web_search(v):
    "Map canonical web_search_options to OpenAI Responses web_search_preview tool."
    t = {"type": "web_search_preview"}
    if (typ := (v or {}).get("type")): t["type"] = typ
    if (s := (v or {}).get("search_context_size")): t["search_context_size"] = s
    if (u := (v or {}).get("user_location")): t["user_location"] = u
    return t

None / empty

In [88]:
test_eq(denorm_web_search(None), {"type": "web_search_preview"})

user_location

In [89]:
loc = {"approximate": {"city": "Istanbul"}}
test_eq(denorm_web_search({"user_location": loc})["user_location"], loc)

### System

In [90]:
#| export
def denorm_system(sp): return sys_text(part_txt(sp))

In [91]:
sp = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
msg1 = mk_user_msg('What are you?')

The Responses request puts the system text in `instructions`.

In [92]:
resp = await oai_call(model=luna, input=denorm_msgs([msg1]), instructions=denorm_system(sp), stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

Arrr, I’m an AI assistant, ready to help ye with questions and tasks.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=31, completion_tokens=22, total_tokens=53, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 31, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 22, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 53})`

:::

### Media Inputs

`InputImage`, `InputAudio`, `InputVideo`, and `InputFile` are shared `Media` subclasses. They store a URL or data URL in `text` and the media type in `mime`. Each provider converter translates these values into its request format.

The media classes describe the input kind, not a promise that every provider or model accepts it. Common formats include JPEG, PNG, GIF, and WebP images; WAV and MP3 audio; MP4 and WebM video; and PDF, DOCX, or TXT files. Check the selected provider and model before sending a format.

The current user-message converters use these representations:

| Part | Responses | Chat | Anthropic | Gemini |
|---|---|---|---|---|
| Image URL or data URL | `input_image.image_url` | `image_url.url` | `image.source` | `inlineData` or `fileData` |
| Base64 audio | Raises `ValueError` | `input_audio` with data and format | Raises `ValueError` | `inlineData` |
| Video URL | `input_video.video_url` | Raises `ValueError` | Raises `ValueError` | `fileData` |
| File URL | `input_file.file_url` | Raises `ValueError` | `document.source` | `fileData` |
| File data URL | `input_file.file_data` and filename | `file.file_data` and filename | `document.source` | `inlineData` |

Responses images use `{"type": "input_image", "image_url": ...}`. Files use `file_url` for URLs or `file_data` plus `filename` for data URLs. The local converter also emits `input_video` for video. It rejects audio.

Chat nests an image URL in `{"type": "image_url", "image_url": {"url": ...}}`. Audio needs a base64 data URL. Its converter emits `{"type": "input_audio", "input_audio": {"data": ..., "format": ...}}` and maps WAV/MP3 MIME types to format names. Files also need base64 data URLs here.

Anthropic uses `{"type": "image", "source": ...}` or `{"type": "document", "source": ...}`. A source contains either a URL or base64 data with `media_type`. The document examples use PDF with `application/pdf`. The converter forwards the supplied MIME type rather than enforcing a PDF-only rule. It applies `cache_control` to supported content blocks.

Gemini uses `{"inlineData": {"mimeType": ..., "data": ...}}` for data URLs. It uses `{"fileData": {"mimeType": ..., "fileUri": ...}}` for URLs, including YouTube video URLs. An `InputVideo` with `start`, `end` or `fps` adds `videoMetadata`, with the offsets as strings in seconds.

The original API comparison cited Responses [input content](https://developers.openai.com/api/reference/resources/responses/methods/create) (`specs/openai.with-code-samples.yml:68361,68429`), Chat [content parts](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create) (`35185,35217,35253`), Anthropic [image/document blocks](https://docs.anthropic.com/en/api/messages) (`specs/anthropic.yml:12159,6740`), and Gemini [Part/Blob/FileData](https://ai.google.dev/api/generate-content) (`specs/gemini.json:189-387`).

In [93]:
#| export
def denorm_user(m:Msg):
    "Convert canonical user Msg to OpenAI Responses input message."
    parts = []
    for p in m.content:
        if   isinstance(p, Text):       parts.append({"type": "input_text", "text": p.text or ""})
        elif isinstance(p, InputImage): parts.append(denorm_image(p))
        elif isinstance(p, InputAudio): raise ValueError("OpenAI Responses API does not support audio input; Coming Soon.") 
        elif isinstance(p, InputVideo): parts.append(denorm_video(p))
        elif isinstance(p, InputFile):  parts.append(denorm_file(p))
    return dict(type='message', role='user', content=parts)

#### Image

In [94]:
#| export
def denorm_image(p): return {"type": "input_image", "image_url": p.text}

Image URL

In [95]:
msg1 = Msg(role='user', content=[InputImage(sample_img_url), Text('What is this image?')])

In [96]:
resp = await oai_call(model=luna, input=denorm_msgs([msg1]), stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

A peaceful mountain lake reflecting the surrounding trees and snowy peaks, with a wooden deck in the foreground.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=588, completion_tokens=24, total_tokens=612, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 588, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 24, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 612})`

:::

Image data

In [97]:
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="
msg1 = Msg(role='user', content=[InputImage(img_b64), Text('What color is this pixel?')])

In [98]:
resp = await oai_call(model=luna, input=denorm_msgs([msg1]), stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

The pixel appears black.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=14, completion_tokens=527, total_tokens=541, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=516, search_queries=0, raw={'input_tokens': 14, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 527, 'output_tokens_details': {'reasoning_tokens': 516}, 'total_tokens': 541})`

:::

#### Audio

This converter rejects `InputAudio`. OpenAI's [Responses migration guide](https://developers.openai.com/api/docs/guides/migrate-to-responses#responses-benefits) lists audio support as forthcoming.

#### Video

In [99]:
#| export
def denorm_video(p): return {"type": "input_video", "video_url": p.text}

#### Files

For a file data URL, `denorm_file` derives a filename such as `upload.pdf` from its MIME type. A file URL uses `file_url` without a filename. Anthropic and Gemini encode the MIME type in their base64 file blocks instead.

In [100]:
#| export
def denorm_file(p):
    if (b64:=data_url(p.text)): return dict(type="input_file", file_data=p.text, filename=f"upload.{b64[0].split('/')[-1]}")
    return {"type": "input_file", "file_url": p.text}

In [101]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
msg1 = Msg(role='user', content=[InputFile(pdf_url), Text('What is the title of this paper?')])

In [102]:
resp = await oai_call(model=luna, input=denorm_msgs([msg1]), stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp

The paper is titled “Attention Is All You Need.”

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=22041, completion_tokens=15, total_tokens=22056, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=0, search_queries=0, raw={'input_tokens': 22041, 'input_tokens_details': {'cache_write_tokens': 22038, 'cached_tokens': 0}, 'output_tokens': 15, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 22056})`

:::

### Media Tool Call Results

A tool result can contain a list of `Text`, `InputImage`, and `InputFile` parts. Other part types raise `ValueError`. Non-list results use `tool_text`. The call ID comes from `id` or `raw['call_id']` and is limited to 64 characters for Codex compatibility.

In [103]:
#| export
def denorm_tool_result(m:ToolResult):
    "Convert canonical tool result back to OpenAI Responses function_call_output item."
    cid = _sanid(m.id or (m.raw or {}).get('call_id'))
    if isinstance(m.text, list):
        out = []
        for p in m.text:
            if   isinstance(p, Text):       out.append({"type": "input_text", "text": p.text or ""})
            elif isinstance(p, InputImage): out.append(denorm_image(p))
            elif isinstance(p, InputFile):  out.append(denorm_file(p))
            else: raise ValueError(f"OpenAI Responses tool_result does not support {p.type}")
        return dict(type='function_call_output', call_id=cid, output=out)
    return dict(type='function_call_output', call_id=cid, output=tool_text(m.text))

In [104]:
def test_img(): "Show the test image"
img_tools = denorm_tool_schs([lite_mk_func(test_img)])
img_q = mk_user_msg('Call test_img, then describe what it shows in one sentence.')
resp = await oai_call(model=mn, input=denorm_msgs([img_q]), tools=img_tools)
comp = mk_completion(resp, mn, api_name, vnd_nm)
msgs = [img_q, comp.message, Msg(role='tool', content=[ToolResult(id=comp.tool_calls[0].id, name='test_img', text=[InputImage(img_b64)])])]
comp.tool_calls


[ToolUse(raw={'id': 'fc_00772f515077fcfe006abb098a698087d0bacbc81c8aabefc1', 'type': 'function_call', 'status': 'completed'}, cache_control=None, id='call_p3FWWiTBOIP0XUeRl8dIS98t', name='test_img', arguments={}, server=False, text=None)]

In [105]:
resp = await oai_call(model=mn, input=denorm_msgs(msgs), tools=img_tools, stream=True)
async for comp in acollect_stream(resp, vendor_name=vnd_nm): pass
comp


A small red-orange vertical shape appears against a black background.

::: details

- model: `None`
- finish_reason: `stop`
- usage: `Usage(prompt_tokens=76, completion_tokens=124, total_tokens=200, cached_tokens=0, cache_creation_tokens=0, reasoning_tokens=106, search_queries=0, raw={'input_tokens': 76, 'input_tokens_details': {'cache_write_tokens': 0, 'cached_tokens': 0}, 'output_tokens': 124, 'output_tokens_details': {'reasoning_tokens': 106}, 'total_tokens': 200})`

:::

## Payload

`mk_payload` translates the shared request options to Responses names. For example, `system` becomes `instructions` and `max_tokens` becomes `max_output_tokens`. It adds web-search options as a tool and retains explicit `False` for `parallel_tool_calls`. `prompt_cache_key` passes through unchanged.

In [106]:
#| export
@delegates(payload_kwargs)
def mk_payload(msgs, model, **kwargs):
    payload = dict(model=model)
    if msgs: payload['input'] = denorm_msgs(msgs)
    if stream:=kwargs.get('stream'):        payload['stream'] = True
    if sp:=kwargs.get('system'):            payload['instructions'] = denorm_system(sp)
    if mt:=kwargs.get('max_tokens'):        payload['max_output_tokens'] = mt
    if tools:=kwargs.get('tools'):          payload['tools'] = denorm_tool_schs(tools)
    if tchc:=kwargs.get('tool_choice'):     payload['tool_choice'] = denorm_tool_choice(tchc)
    if rid:=kwargs.get('previous_response_id'): payload['previous_response_id'] = rid
    if key:=kwargs.get('prompt_cache_key'): payload['prompt_cache_key'] = key
    if (ptc:=kwargs.get('parallel_tool_calls')) is not None: payload['parallel_tool_calls'] = ptc
    if thk:=kwargs.get('reasoning_effort'): payload['reasoning'] = denorm_reasoning(thk)
    if (wopts:=kwargs.get('web_search_options')) is not None:  payload.setdefault('tools', []).append(denorm_web_search(wopts))
    if (temp:=kwargs.get('temperature')) is not None: payload['temperature'] = temp
    return payload

`get_hdrs` prefers `oauth_token` for bearer authentication. Otherwise it uses the explicit API key or `OPENAI_API_KEY`.

In [107]:
#| export
def get_hdrs(api_key=None, oauth_token=None):
    "Bearer auth: an OAuth access token, or else an API key from the argument or the environment"
    return {"Authorization": f"Bearer {oauth_token or get_api_key(api_key, 'OPENAI_API_KEY')}"}


## Cost

`cost` uses the response's raw token usage and model metadata from `get_model_info`. It selects the input-length price tier, charges uncached input and output tokens, then adds cached-input charges.

In [108]:
#| export
def cost(usage, m):
    raw = usage.raw
    cached = raw.get('input_tokens_details', {}).get('cached_tokens', 0)
    tier = price_tier(m, raw['input_tokens'])
    cost  = (raw['input_tokens'] - cached) * tier_rate(m, 'input_cost_per_token', tier)
    cost += raw['output_tokens'] * tier_rate(m, 'output_cost_per_token', tier)
    cost += cached * tier_rate(m, 'cache_read_input_token_cost', tier)
    return cost

## Register API

The final registry entry supplies the normalizers, payload builder, stream collector, authentication headers, cost function, and `/responses` endpoint. `raw_msg` and `collate_raw` retain replay data. `supports_previous_response_id` advertises provider continuation support. `acomplete` uses this entry to execute requests.

`fix_payload` modifies the completed request before sending it. For `gpt-6-astra`, it removes `temperature`. For the Codex vendor, it removes sampling, token-limit, and metadata fields and sets `store=False`.

In [109]:
#| export
def fix_payload(payload, model, vendor_name):
    "Apply vendor request quirks to a built payload."
    if model == 'gpt-6-astra': payload.pop('temperature', None)
    if vendor_name == 'codex':
        for k in 'temperature max_tokens max_output_tokens max_completion_tokens metadata'.split(): payload.pop(k, None)
        payload['store'] = False

In [110]:
p = dict(model='m', temperature=0.5, store=True)
fix_payload(p, luna, 'codex')
test_eq(p, dict(model='m', store=False))
p2 = dict(model='m', temperature=0.5)
fix_payload(p2, luna, None)
test_eq(p2, dict(model='m', temperature=0.5))
p3 = mk_payload([], 'gpt-6-astra', temperature=0.5, reasoning_effort='low')
fix_payload(p3, 'gpt-6-astra', 'openai')
assert 'temperature' not in p3
p3

{'model': 'gpt-6-astra', 'reasoning': {'effort': 'low'}}

In [111]:
#| export
api_ns = dict(norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    supports_previous_response_id=True, raw_msg=raw_msg, collate_raw=collate_raw, acollect_stream=acollect_stream, mk_payload=mk_payload,
    cost=cost, get_hdrs=get_hdrs, fix_payload=fix_payload, endpoint='/responses')
api_registry.register('openai', **api_ns)

Hosted web searches add a per-query charge to the token cost. `norm_usage` records their count in `Usage.search_queries`. The shared `Completion.cost` property multiplies that count by `search_context_cost_per_query['search_context_size_medium']` from model metadata. This transport's `cost` function calculates tokens only.

The price metadata in this example gives the same search rate for each context size. The shared calculation uses the medium rate regardless of the requested size.

In [112]:
tokens_only = cost(c.usage, get_model_info(mn, vnd_nm))
test_close(c.cost - tokens_only, c.usage.search_queries * 0.01)
c.usage.search_queries, tokens_only, c.cost

(0, 3.4999999999999995e-06, 3.4999999999999995e-06)

## Costs through `acomplete`

`acomplete` sends `gpt` and `o`-series models through this adapter. These checks compare `Completion.cost` between streamed and non-streamed `acomplete` calls.

Text

In [113]:
msg = mk_user_msg("Say hello in French")

In [114]:
comp = await acomplete([msg], model=luna, max_tokens=64)
comp.cost

4e-06

In [115]:
# Streaming - as a sanity check
it = await acomplete([msg], model=luna, max_tokens=64, stream=True)
async for scomp in it: pass
test_eq(scomp.usage.prompt_tokens, comp.usage.prompt_tokens)
assert scomp.cost > 0

Image

In [116]:
msg = Msg(role='user', content=[InputImage(sample_img_url), Text('What is this image?')])
comp = await acomplete([msg], model=luna, max_tokens=64)

In [117]:
# Streaming - as a sanity check
it = await acomplete([msg], model=luna, max_tokens=64, stream=True)
async for scomp in it: pass
test_eq(scomp.usage.prompt_tokens, comp.usage.prompt_tokens)
assert scomp.cost > 0

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev
nbdev.nbdev_export()